In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
df = spark.read.csv("/Volumes/datanotfound/main_data/data/bank_data.csv", header=True, inferSchema=True)
df.show()
display(df.count())
df.printSchema()
df1=spark.read.csv("/Volumes/datanotfound/main_data/data/customer_data.csv", header=True, inferSchema=True)
df1.show()
print(df1.count())
df2=spark.read.csv("/Volumes/datanotfound/main_data/data/transaction_data_new.csv", header=True, inferSchema=True)
df2.show()
print(df2.count())

In [0]:
df.printSchema()
df1.printSchema()
df2.printSchema()

In [0]:
display(df.dtypes)
display(df1.dtypes)
display(df2.dtypes)

In [0]:
from pyspark.sql.functions import col, when, count

df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()



In [0]:
df1.select([count(when(col(c).isNull(), c)).alias(c) for c in df1.columns]).show()



In [0]:


df2.select([count(when(col(c).isNull(), c)).alias(c) for c in df2.columns]).show()

In [0]:
df.filter(col("Profit_Margin") < 0).show()



In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [c for c, t in df.dtypes if t in ('int', 'double', 'float', 'bigint')]

df.select([
    count(when(col(c) < 0, True)).alias(c)
    for c in numeric_cols
]).show()

In [0]:
df.groupBy("Branch_ID").count().filter(col("count") > 1).show()

In [0]:
df1.show()

In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [c for c, t in df1.dtypes if t in ('int', 'double', 'float', 'bigint')]

df1.select([
    count(when(col(c) < 0, True)).alias(c)
    for c in numeric_cols
]).show()

In [0]:
df1.groupBy("Customer_ID").count().filter(col("count") > 1).show()

In [0]:

df2.filter(col("Transaction_Amount") < 0).show()


In [0]:
df2.filter(col("Total_Balance") < 0).show()


In [0]:
df2.filter(col("Transaction_Amount") < 0).show()


In [0]:
df2.filter(col("Investment_Amount") < 0).show()

In [0]:
df2.groupBy("Transaction_ID").count().filter(col("count") > 1).show()

In [0]:
df2.groupBy("Investment_Type").count().filter(col("count") > 1).show()

In [0]:
df2.groupBy("Account_Type").count().filter(col("count") > 1).show()

In [0]:
from pyspark.sql.types import NumericType

numeric_cols = [c for c, t in df2.dtypes if t in ('int', 'double', 'float', 'bigint')]

df2.select([
    count(when(col(c) < 0, True)).alias(c)
    for c in numeric_cols
]).show()

In [0]:
from pyspark.sql.functions import lower, trim

df2.select([
    count(
        when(
            col(c).isNull() | 
            (trim(col(c)) == "") | 
            (lower(col(c)) == "na") | 
            (lower(col(c)) == "null"),
            c
        )
    ).alias(c)
    for c in df2.columns
]).show()

In [0]:
df1.write.format("delta").mode("overwrite").save("/Volumes/datanotfound/bronze/raw_data1")
df2.write.format("delta").mode("overwrite").save("/Volumes/datanotfound/bronze/raw_data2")

In [0]:
Bank_data=spark.read.format("delta").load("/Volumes/datanotfound/bronze/raw_data")

Bank_data.show()